# Wavelet subsample — uniform per session (paw, left paw, wheel, left paw + wheel)

Copy of `3.2_wavelet_subsample.ipynb` with the training-set construction changed. The
original is left untouched.

**What changed**

| | `3.2_wavelet_subsample` | this notebook |
|---|---|---|
| frames kept per session | 2,000 of 20,000, drawn with p ∝ KDE density of a per-session t-SNE | 2,000 **uniformly at random** from all valid frames |
| features | 32 columns (raw positions + 0.5–32 Hz) for the t-SNE; 3.3 then clusters on 20 | only the 20 columns 3.3 clusters on |
| normalisation | subsample z-scored by its **own** stats | nothing applied; the **full-session** stats (raw and log) are saved so 3.3 applies exactly what it later applies to every frame |

**Why not the density weighting.** Berman et al. 2014 (*J R Soc Interface*, "Mapping the
stereotyped behaviour of freely moving fruit flies") run t-SNE on 20,000 random points per
fly, watershed the density, and draw from each region **in proportion to the integral of the
density over that region**: stratified sampling that *preserves* each individual's
composition while making sure every region is represented. Drawing individual points with
p ∝ density instead weights them by density twice (they already occur in proportion to it).
On 25 sessions that made the training set's cluster make-up differ from the session's by a
total variation of 0.09, against 0.02 for a uniform draw, with the rarest vigorous cluster
at ×0.71. A uniform draw is Berman's scheme in expectation, without the t-SNE.

Every session still contributes the same number of frames, so a long session does not
outvote a short one.

**What it can subsample** (`DATASET`, `VAR`; definitions in the first code cell):
- `DATASET`: `'proficient'` (`data/`), `'neuromodulators'` (`data/neuromodulators/`),
  or `'training'` (`data/training/`).
- `VAR`: `'paw'` (both paws, 20 features), `'left_paw'` (10), `'wheel'` (5), or
  `'left_paw_wheel'` (15: left paw and wheel wavelets merged on `Bin`). This replaces
  `3.2_paw_wheel_subsample.ipynb`.

Wheel uses 0.5–8 Hz like the paw. The original 3.3's wheel-only setting used 4–32 Hz.

In [ ]:
"""SHARED DEFINITIONS -- live in wavelet_clusters.py (shared by 3.2_..._uniform, 3.3_..._uniform and paw_k_sweep.py)."""
import os
import sys
import pathlib
import numpy as np
import pandas as pd

_seg = pathlib.Path.cwd().resolve()
while not (_seg / 'wavelet_clusters.py').exists() and _seg != _seg.parent:
    _seg = _seg / 'segmentation' if (_seg / 'segmentation' / 'wavelet_clusters.py').exists() else _seg.parent
sys.path.insert(0, str(_seg))
import wavelet_clusters as wc
from wavelet_clusters import (DATA, DATASETS, BANDS, SOURCES, VARS, STAGE_DIR, features, feature_sources,
                              feature_eps, source_dir, session_tags, load_frame, load_session, stage_eids,
                              dataset_root)

In [4]:
"""KNOBS"""
DATASET = 'proficient'  # 'proficient' | 'neuromodulators' | 'training'
VAR     = 'left_paw_wheel'         # 'paw' | 'left_paw' | 'wheel' | 'left_paw_wheel'
N_SUB   = 2000          # frames per session -- every session weighs the same
SEED    = 0
MAX_SESSIONS = None     # an int runs only the first n sessions, for a quick test
OUT_ROOT     = None     # None = DATA; a path redirects the output (tests), same layout

OUT = dataset_root(DATASET, OUT_ROOT) + f'{VAR}_subsampled_wavelets_uniform/'
os.makedirs(OUT, exist_ok=True)
tags = session_tags(DATASET, VAR)[:MAX_SESSIONS]
EPS = feature_eps(VAR)          # the log floor of each feature's source (see SOURCES)
print(f'{DATASET}, {VAR}: {len(tags)} sessions, {len(features(VAR))} features -> {OUT}')

proficient, left_paw_wheel: 332 sessions, 15 features -> /home/ines/repositories/representation_learning_variability/paper-individuality/data/left_paw_wheel_subsampled_wavelets_uniform/


In [5]:
"""SUBSAMPLE -- one .npz per session."""
rng = np.random.default_rng(SEED)
too_short = []
for tag in tags:
    X, bins, _ = load_session(DATASET, VAR, tag)
    if len(X) < N_SUB:
        too_short.append((tag, len(X)))
        continue
    pick = np.sort(rng.choice(len(X), N_SUB, replace=False))
    L = np.log(X + EPS)
    np.savez(OUT + tag, sub=X[pick], bins=bins[pick], features=np.array(features(VAR)),
             n_valid=len(X), eps=EPS,
             mean_raw=X.mean(0), sd_raw=X.std(0),       # ddof=0, as scipy.stats.zscore
             mean_log=L.mean(0), sd_log=L.std(0))
print(f'saved {len(tags) - len(too_short)}; too short: {too_short}')

saved 332; too short: []
